# Đánh giá M4 trên validation

So sánh M1, M3 và M4 trên cùng 1.407 câu, cùng cấu hình suy luận.
Notebook 09 giữ output huấn luyện. Kết quả mới lưu riêng; không dùng test để chọn mô hình.


In [ ]:
import hashlib
import json
import math
import os
import re
import runpy
import subprocess
import sys
from pathlib import Path

LAUNCH_RECORD = 'source/answer_preservation_validation_seed42_20261105.json'
LAUNCH_SHA256 = ''
PROJECT_DIR = os.environ.get('VQA_PROJECT_ROOT')
RUN_EXPORT = False
RUN_SCORE = False

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


## 1. Nguồn và dữ liệu

Chạy lần lượt các ô setup. Chỉ tiếp tục khi báo cáo M4, adapter và dữ liệu khớp bản đã ghim.


In [ ]:
from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive', timeout_ms=600000)
if PROJECT_DIR:
    ROOT = Path(PROJECT_DIR)
else:
    candidates = [Path('/content/drive/MyDrive/Vietnamese_VQA_NLE')]
    candidates += list(Path('/content/drive/.shortcut-targets-by-id').glob('*/Vietnamese_VQA_NLE'))
    roots = {p.resolve() for p in candidates if (p / LAUNCH_RECORD).is_file()}
    if len(roots) != 1:
        raise ValueError('Set PROJECT_DIR to the mounted project folder')
    ROOT = roots.pop()
if not re.fullmatch(r'[0-9a-f]{64}', LAUNCH_SHA256):
    raise ValueError('Set the pinned launch SHA-256')
if sha256(ROOT / LAUNCH_RECORD) != LAUNCH_SHA256:
    raise ValueError('Validation launch changed')
launch = json.loads((ROOT / LAUNCH_RECORD).read_text('utf-8'))
if (launch['schema'] != 'answer_preservation_validation_v1'
        or launch['split'] != 'validation' or launch['questions'] != 1407):
    raise ValueError('Require the pinned full-validation launch')
print('Project:', ROOT)


In [ ]:
bootstrap = ROOT / 'source/bootstrap_cloud_source.py'
if sha256(bootstrap) != '391172ed1fbf39d0eeeef2c9aa3d056d7770085d0a10574d571470d7f2bbef40':
    raise ValueError('Bootstrap checksum mismatch')
release = launch['source_release']
if not re.fullmatch(r'source_[A-Za-z0-9_]+', release):
    raise ValueError('Invalid source release')
REPO = Path(subprocess.check_output([
    sys.executable, str(bootstrap), '--archive',
    str(ROOT / 'source/releases' / f'{release}.zip'),
    '--sha256', launch['source_sha256'], '--work-root', '/content/vqa_sources',
], text=True).strip())
run_command = runpy.run_path(str(REPO / 'src/vqa_explanation/utils/process.py'))['run_command']

def run(command):
    run_command([str(value) for value in command], cwd=REPO)

ENV_DIR = Path('/content/vqa_envs') / release
PYTHON = ENV_DIR / 'bin/python'
run([sys.executable, REPO / 'scripts/setup_preference_environment.py',
     '--environment-dir', ENV_DIR])
print('Source:', release)


In [ ]:
report_path = ROOT / launch['training_result']
if sha256(report_path) != launch['training_result_sha256']:
    raise ValueError('M4 training report changed')
report = json.loads(report_path.read_text('utf-8'))
required = {'status': 'passed', 'global_step': 375, 'epoch': 3.0,
            'objective': 'dpo_answer_nll', 'answer_nll_weight': 0.1,
            'policy_updated': True, 'reference_unchanged': True,
            'saved_policy_verified': True, 'plan_sha256': launch['training_plan_sha256']}
if any(report.get(key) != value for key, value in required.items()):
    raise ValueError('M4 did not complete the pinned training protocol')
if not math.isfinite(report['training_loss']):
    raise ValueError('Nonfinite training loss')
if report['output_adapter']['files'] != launch['adapter_files']['m4']:
    raise ValueError('M4 adapter identity differs from training report')
for arm, relative in launch['adapters'].items():
    for filename, expected in launch['adapter_files'][arm].items():
        if sha256(ROOT / relative / filename) != expected:
            raise ValueError(f'{arm}: adapter checksum mismatch')
print('M4:', report['global_step'], 'updates;', report['epoch'], 'epochs')
print('M1/M3/M4 adapter checksums verified')


In [ ]:
DATA = Path('/content/vqa_work/answer_preservation_validation')
run([PYTHON, REPO / 'scripts/stage_preference_data.py', 'restore',
     '--archive', ROOT / launch['subset_archive'],
     '--sha256', launch['subset_sha256'], '--destination', DATA])
VALIDATION = DATA / 'metadata/validation.jsonl'
if sha256(VALIDATION) != launch['validation_sha256']:
    raise ValueError('Canonical validation annotations changed')
references = [json.loads(line) for line in VALIDATION.read_text('utf-8').splitlines() if line]
if (len(references) != 1407 or len({r['question_id'] for r in references}) != 1407
        or len({r['image_id'] for r in references}) != 1383
        or any(r['split'] != 'validation' for r in references)):
    raise ValueError('Incomplete validation references')
OUT = ROOT / launch['output_dir']
print('Validation:', len(references), 'questions;', 1383, 'images')
print('Output:', OUT)


## 2. Sinh đáp án và giải thích

Mỗi nhánh chạy trong tiến trình riêng. Export dở được tiếp tục khi manifest khớp;
không trộn prediction cũ khác cấu hình vào lượt này.


In [ ]:
if RUN_EXPORT:
    for arm in ('m4', 'm3', 'm1'):
        output = OUT / f'{arm}.jsonl'
        command = [PYTHON, REPO / 'scripts/run_m0.py',
                   '--annotations', DATA / 'metadata/all.jsonl', '--image-root', DATA,
                   '--output', output, '--split', 'validation', '--limit', 0,
                   '--seed', launch['seed'], '--max-new-tokens', launch['max_new_tokens'],
                   '--min-pixels', launch['pixels'], '--max-pixels', launch['pixels'],
                   '--load-in-4bit', '--adapter', ROOT / launch['adapters'][arm],
                   '--experiment-id', launch['experiments'][arm],
                   '--loading-profile', REPO / launch['loading_profile']]
        if output.exists() or output.with_suffix('.jsonl.manifest.json').exists():
            command.append('--resume')
        run(command)
else:
    print('Export not requested')


## 3. Chỉ số và so sánh cặp

Tính EM, F1, ROUGE-L và BERTScore. Khoảng tin cậy lấy mẫu lại theo ảnh.
Các chỉ số văn bản không thay việc chấm giải thích trên ảnh.


In [ ]:
if RUN_SCORE:
    SEMANTIC_DIR = Path('/content/vqa_envs') / f'{release}_semantic'
    setup = (
        'import sys; from pathlib import Path; '
        'sys.path.insert(0, str(Path(sys.argv[1]) / "src")); '
        'from vqa_explanation.utils.cloud_environment import prepare_environment; '
        'prepare_environment(Path(sys.argv[1]), Path(sys.argv[2]), profile="semantic")'
    )
    run([sys.executable, '-c', setup, REPO, SEMANTIC_DIR])
    SEMANTIC_PYTHON = SEMANTIC_DIR / 'bin/python'
    for arm in ('m4', 'm3', 'm1'):
        predictions = OUT / f'{arm}.jsonl'
        metrics_path = OUT / f'{arm}_metrics.json'
        if metrics_path.is_file():
            metrics = json.loads(metrics_path.read_text('utf-8'))
            if (metrics.get('predictions_sha256') != sha256(predictions)
                    or metrics.get('n') != 1407 or metrics.get('split') != 'validation'
                    or metrics.get('explanation_bertscore', {}).get('status') != 'scored'):
                raise ValueError('Existing metrics do not match complete predictions')
        else:
            run([SEMANTIC_PYTHON, REPO / 'scripts/evaluate.py', predictions,
                 '--split', 'validation', '--output', metrics_path,
                 '--bertscore', '--bertscore-device', 'cuda', '--bertscore-batch-size', 16])
else:
    print('Scoring not requested')


In [ ]:
if RUN_SCORE:
    comparison_paths = {arm: OUT / f'm4_minus_{arm}.json' for arm in ('m3', 'm1')}
    comparison_inputs = {arm: {name: sha256(OUT / f'{arm}{name}')
                              for name in ('.jsonl', '_metrics.json')}
                         for arm in ('m1', 'm3', 'm4')}
    pin = OUT / 'comparison_inputs.json'
    if pin.exists():
        if json.loads(pin.read_text('utf-8')) != comparison_inputs:
            raise ValueError('Comparison inputs changed; use a new output directory')
    else:
        if any(path.exists() for path in comparison_paths.values()):
            raise ValueError('Existing comparisons have no input receipt')
        with pin.open('x', encoding='utf-8') as stream:
            json.dump(comparison_inputs, stream, indent=2)
    for arm, comparison_path in comparison_paths.items():
        if not comparison_path.exists():
            run([SEMANTIC_PYTHON, REPO / 'scripts/compare_runs.py',
                 OUT / f'{arm}.jsonl', OUT / 'm4.jsonl', '--output', comparison_path,
                 '--expected-annotations', VALIDATION,
                 '--expected-sha256', launch['validation_sha256'], '--split', 'validation',
                 '--left-experiment', launch['experiments'][arm],
                 '--right-experiment', launch['experiments']['m4'],
                 '--bootstrap-resamples', 2000, '--bootstrap-seed', 42,
                 '--left-semantic-report', OUT / f'{arm}_metrics.json',
                 '--right-semantic-report', OUT / 'm4_metrics.json'])
        result = json.loads(comparison_path.read_text('utf-8'))
        if result.get('comparable') is not True or result.get('n') != 1407:
            raise ValueError('Runs are not comparable')
        print(f'M4 minus {arm.upper()}:', result['delta_right_minus_left'])


In [ ]:
if RUN_SCORE:
    print('Model | EM | F1 | ROUGE-L | BERTScore (percent)')
    for arm in ('m1', 'm3', 'm4'):
        metrics = json.loads((OUT / f'{arm}_metrics.json').read_text('utf-8'))
        values = [metrics['exact_match'], metrics['token_f1'],
                  metrics['explanation_rouge_l_f1'],
                  metrics['explanation_bertscore']['mean_f1']]
        print(arm.upper(), '|', ' | '.join(f'{100 * value:.2f}' for value in values))
    print('Validation only; image-based quality assessment remains separate.')
